In [ ]:
import pandas as pd
seq_regions = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
cf_regions = ['CF_H1', 'CF_H2', 'CF_L1', 'CF_L2', 'CF_L3']
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset=seq_regions)
    .dropna(subset=cf_regions)
    .drop_duplicates(subset=seq_regions) 
)

In [ ]:
import pandas as pd
import os

# CDR-Regionen definieren
cdrs = ["H1", "H2", "L1", "L2", "L3"]

# Mapping zwischen CF-Spalten und neuen HC-Spalten
cf_columns = [f"CF_{cdr}" for cdr in cdrs]
hc_columns = [f"HC_{cdr}" for cdr in cdrs]



df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset=seq_regions)
    .dropna(subset=cf_regions)
    .drop_duplicates(subset=seq_regions)
    .reset_index(drop=True)
)

# Lege leere HC-Spalten an
for hc_col in hc_columns:
    df[hc_col] = None

# Füge pro CDR-Typ die Clusterlabels hinzu
for cdr in cdrs:
    cluster_df = pd.read_csv(f"cdr_cluster_tsvs/clusters_SEQ_{cdr}.tsv", sep="\t")

    # Mapping: pdb_id → cluster_label
    cluster_map = dict(zip(cluster_df["pdb"], cluster_df["Cluster_Label"]))

    # Schreibe ins Haupt-DataFrame
    df[f"HC_{cdr}"] = df["pdb"].map(cluster_map)

# Speichern als neue Datei
df.to_csv("ab_ag_hierarchical_canonical_forms.csv", index=False)



In [5]:
# v measure ohne länge
import pandas as pd
from sklearn.metrics import v_measure_score

# Dateien einlesen
df_true = df  # Original
df_pred = pd.read_csv("ab_ag_hierarchical_canonical_forms.csv")  # Hierarchische Cluster

# CDRs, die verglichen werden sollen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

V-Measure Ergebnisse:

CDR H1: V-Measure = 0.110
CDR H2: V-Measure = 0.149
CDR L1: V-Measure = 0.359
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.301


In [6]:
import pandas as pd
from sklearn.metrics import v_measure_score

# Dateien einlesen
df_true =df  # Original
df_pred = pd.read_csv("ab_ag_hierarchical_canonical_forms_nach_länge.csv")  # neue Datei mit Länge-Clustern

# CDRs, die verglichen werden sollen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

V-Measure Ergebnisse:

CDR H1: V-Measure = 0.367
CDR H2: V-Measure = 0.476
CDR L1: V-Measure = 0.718
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.589
